In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Literal
import json
import numpy as np
from dataclasses import dataclass
import warnings
import os
from langchain_naver import ChatClovaX
from langchain_core.messages import HumanMessage, SystemMessage
warnings.filterwarnings('ignore')

import dotenv
dotenv.load_dotenv()

# @dataclass
# class CompanyInfo:
#     """기업 정보 데이터 클래스"""
#     name: str
#     ticker: str = None
#     is_listed: bool = None
#     market: str = None  # KOSPI, KOSDAQ, PRIVATE
#     sector: str = None

# @dataclass
# class FinancialData:
#     """재무 데이터 클래스"""
#     revenue: float = None
#     operating_income: float = None
#     net_income: float = None
#     total_assets: float = None
#     total_equity: float = None
#     shares_outstanding: float = None
#     market_cap: float = None
#     current_price: float = None

# @dataclass
# class ValuationRatios:
#     """밸류에이션 비율 클래스"""
#     per: float = None
#     pbr: float = None
#     ev_ebitda: float = None
#     roe: float = None
#     debt_ratio: float = None

class LLMAgent:
    """LLM 에이전트 클래스"""
    
    def __init__(self, api_key: str = None):
        """
        LLM 초기화
        api_key가 없으면 환경변수에서 가져옴
        """
        if not api_key:
            api_key = os.getenv('NCP_CLOVASTUDIO_API_KEY')
        
        if not api_key:
            raise ValueError("API 키를 설정해주세요. 환경변수 NCP_CLOVASTUDIO_API_KEY 또는 직접 전달")
        
        self.llm = ChatClovaX(
            api_key=api_key,
            model="HCX-005",
            temperature=0.3,
            max_tokens=4000,
        )
    
    def analyze_query(self, query: str) -> Dict[str, str]:
        """사용자 질문을 분석하여 기업명과 의도 추출"""
        
        system_prompt = """
        당신은 금융 투자 전문가입니다. 사용자의 질문을 분석하여 다음 정보를 JSON 형태로 추출해주세요:
        
        1. company_name: 질문에서 언급된 기업명 (정확한 기업명으로 표준화)
        2. intent: 질문의 의도 (INVESTMENT_ADVICE, SELL_ADVICE, ANALYSIS, GENERAL 중 하나)
        3. specific_concern: 사용자가 특별히 궁금해하는 부분 (재무상태, 성장성, 리스크 등)
        
        한국의 주요 기업명은 다음과 같이 표준화해주세요:
        - 삼성전자, SK하이닉스, LG에너지솔루션, 카카오, 네이버, 현대차, LG화학
        - 무신사, 컬리, 토스, 직방, 야놀자 (비상장)
        
        응답은 반드시 유효한 JSON 형태로만 해주세요.
        """
        
        try:
            response = self.llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=query)
            ])
            
            # JSON 파싱 시도
            result = json.loads(response.content)
            return result
        except:
            # 파싱 실패시 기본값 반환
            return {
                "company_name": None,
                "intent": "GENERAL",
                "specific_concern": "전반적인 투자 가치"
            }
    
    def generate_investment_report(self, query: str, company_info: Dict, financial_data: Dict, market_context: Dict) -> str:
        """수집된 정보를 바탕으로 투자 리포트 생성"""
        
        # 데이터를 LLM이 이해하기 쉬운 형태로 정리
        data_summary = self._format_data_for_llm(company_info, financial_data, market_context)
        
        system_prompt = f"""
        당신은 경험이 풍부한 증권 애널리스트입니다. 다음 데이터를 분석하여 전문적이고 객관적인 투자 리포트를 작성해주세요.

        **분석 가이드라인:**
        1. 객관적 사실 기반으로 분석
        2. 리스크와 기회 요인 균형있게 제시
        3. 명확한 투자 판단 제시 (매수/보류/매도 또는 투자권장/주의)
        4. 구체적인 근거와 수치 제시
        5. 한국어로 작성

        **리포트 구조:**
        ## [기업명] 투자 분석 리포트

        ### 📊 기본 현황
        - 현재 상황 요약

        ### 💰 재무 분석
        - 주요 재무지표 분석
        - 업계 평균과의 비교

        ### ⚖️ 투자 판단: [매수/보류/매도]
        
        ### 📈 투자 포인트
        - 긍정적 요인들
        
        ### ⚠️ 리스크 요인
        - 주의해야 할 점들

        ### 🎯 결론
        - 최종 투자 의견 및 이유

        **데이터:**
        {data_summary}
        """
        
        try:
            response = self.llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=f"사용자 질문: {query}")
            ])
            
            return response.content
        except Exception as e:
            return f"리포트 생성 중 오류가 발생했습니다: {str(e)}"
    
    def _format_data_for_llm(self, company_info: Dict, financial_data: Dict, market_context: Dict) -> str:
        """데이터를 LLM이 분석하기 쉬운 텍스트 형태로 변환"""
        
        formatted_data = f"""
        [기업 기본정보]
        - 기업명: {company_info.get('name', 'N/A')}
        - 상장여부: {'상장기업' if company_info.get('is_listed') else '비상장기업'}
        - 시장: {company_info.get('market', 'N/A')}
        - 업종: {company_info.get('sector', 'N/A')}
        """
        
        if company_info.get('is_listed'):
            # 상장기업 데이터
            formatted_data += f"""
        [재무정보]
        - 현재 주가: {financial_data.get('current_price', 'N/A'):,.0f}원
        - 시가총액: {financial_data.get('market_cap', 0)/1000000000000:.1f}조원
        - 매출액: {financial_data.get('revenue', 0)/1000000000000:.1f}조원
        - 순이익: {financial_data.get('net_income', 0)/1000000000000:.1f}조원
        
        [밸류에이션 지표]
        - PER: {financial_data.get('per', 'N/A')}
        - PBR: {financial_data.get('pbr', 'N/A')}
        - ROE: {financial_data.get('roe', 'N/A')}%
        - 부채비율: {financial_data.get('debt_ratio', 'N/A')}%
        
        [시장 비교]
        - 업계 평균 PER: {market_context.get('industry_per', 'N/A')}
        - 업계 평균 PBR: {market_context.get('industry_pbr', 'N/A')}
        - 업계 평균 ROE: {market_context.get('industry_roe', 'N/A')}%
        """
        else:
            # 비상장기업 데이터
            formatted_data += f"""
        [기업가치 정보]
        - 최근 기업가치: {financial_data.get('current_valuation', 0)/1000000000000:.1f}조원
        - 추정 매출: {financial_data.get('revenue_estimate', 0)/1000000000000:.1f}조원
        - 최근 투자유치: {financial_data.get('last_funding', 'N/A')}
        - 임직원 수: {financial_data.get('employees', 'N/A')}명
        
        [업종 분석]
        - 업종 멀티플: {market_context.get('sector_multiple', 'N/A')}배
        - 업종 성장률: {market_context.get('growth_rate', 'N/A')}%
        - 시장 전망: {market_context.get('market_outlook', '보통')}
        """
        
        # 최신 뉴스나 이슈가 있다면 추가
        if market_context.get('recent_news'):
            formatted_data += f"""
        [최근 이슈]
        {market_context['recent_news']}
        """
        
        return formatted_data

class NLPAnalyzer:
    """LLM 기반 자연어 처리 및 기업 분류 모듈"""
    
    def __init__(self, llm_agent: LLMAgent):
        self.llm_agent = llm_agent
        # 기업 데이터베이스
        self.listed_companies = {
            '삼성전자': {'ticker': '005930', 'market': 'KOSPI', 'sector': '반도체'},
            'SK하이닉스': {'ticker': '000660', 'market': 'KOSPI', 'sector': '반도체'},
            'LG에너지솔루션': {'ticker': '373220', 'market': 'KOSPI', 'sector': '배터리'},
            '카카오': {'ticker': '035720', 'market': 'KOSDAQ', 'sector': '인터넷'},
            '네이버': {'ticker': '035420', 'market': 'KOSPI', 'sector': '인터넷'},
            'NAVER': {'ticker': '035420', 'market': 'KOSPI', 'sector': '인터넷'},
            '현대차': {'ticker': '005380', 'market': 'KOSPI', 'sector': '자동차'},
            'LG화학': {'ticker': '051910', 'market': 'KOSPI', 'sector': '화학'},
        }
        
        self.unlisted_companies = {
            '무신사': {'sector': '패션 커머스'},
            '컬리': {'sector': '식품 커머스'},
            '토스': {'sector': '핀테크'},
            '직방': {'sector': '부동산테크'},
            '야놀자': {'sector': '여행테크'},
        }
    
    def analyze_query(self, query: str) -> Dict:
        """LLM을 사용한 쿼리 분석"""
        analysis = self.llm_agent.analyze_query(query)
        return analysis
    
    def classify_company(self, company_name: str) -> CompanyInfo:
        """기업 분류"""
        if company_name in self.listed_companies:
            info = self.listed_companies[company_name]
            return CompanyInfo(
                name=company_name,
                ticker=info['ticker'],
                is_listed=True,
                market=info['market'],
                sector=info['sector']
            )
        elif company_name in self.unlisted_companies:
            info = self.unlisted_companies[company_name]
            return CompanyInfo(
                name=company_name,
                is_listed=False,
                market='PRIVATE',
                sector=info.get('sector')
            )
        else:
            return CompanyInfo(name=company_name, is_listed=None)

class ListedCompanyDataCollector:
    """상장기업 데이터 수집기"""
    
    def get_yahoo_data(self, ticker: str) -> Dict:
        """Yahoo Finance에서 데이터 수집"""
        try:
            yf_ticker = f"{ticker}.KS"
            stock = yf.Ticker(yf_ticker)
            info = stock.info
            hist = stock.history(period="1y")
            
            if hist.empty:
                yf_ticker = f"{ticker}.KQ"
                stock = yf.Ticker(yf_ticker)
                info = stock.info
                hist = stock.history(period="1y")
            
            current_price = hist['Close'].iloc[-1] if not hist.empty else None
            
            return {
                'current_price': current_price,
                'market_cap': info.get('marketCap'),
                'pe_ratio': info.get('trailingPE'),
                'pb_ratio': info.get('priceToBook'),
                'revenue': info.get('totalRevenue'),
                'shares_outstanding': info.get('sharesOutstanding'),
                'sector': info.get('sector'),
                'industry': info.get('industry')
            }
        except Exception as e:
            print(f"Yahoo Finance 데이터 수집 오류: {e}")
            return {}
    
    def get_financial_ratios(self, financial_data: Dict) -> Dict:
        """재무비율 계산"""
        ratios = {}
        
        try:
            # PER은 Yahoo에서 직접 가져오거나 계산
            ratios['per'] = financial_data.get('pe_ratio')
            ratios['pbr'] = financial_data.get('pb_ratio')
            
            # 기본 재무비율들 (실제로는 더 정교한 계산 필요)
            if financial_data.get('revenue') and financial_data.get('market_cap'):
                ratios['price_to_sales'] = financial_data['market_cap'] / financial_data['revenue']
            
        except Exception as e:
            print(f"재무비율 계산 오류: {e}")
        
        return ratios
    
    def collect_data(self, company_info: CompanyInfo) -> Dict:
        """상장기업 종합 데이터 수집"""
        yahoo_data = self.get_yahoo_data(company_info.ticker)
        ratios = self.get_financial_ratios(yahoo_data)
        
        # 모든 데이터를 하나의 딕셔너리로 통합
        combined_data = {
            **yahoo_data,
            **ratios,
            'net_income': yahoo_data.get('revenue', 0) * 0.1,  # 예시: 매출의 10%로 가정
            'roe': 10.5,  # 예시 데이터
            'debt_ratio': 45.2  # 예시 데이터
        }
        
        return combined_data

# class UnlistedCompanyDataCollector:
#     """비상장기업 데이터 수집기"""
    
#     def get_company_info(self, company_name: str) -> Dict:
#         """비상장기업 정보 수집"""
#         company_profiles = {
#             '무신사': {
#                 'current_valuation': 3000000000000,  # 3조원
#                 'last_funding': '2023-05',
#                 'employees': 3000,
#                 'revenue_estimate': 800000000000,  # 8천억원
#                 'recent_news': '패션 플랫폼 시장에서 독보적 위치 확보, 해외 진출 가속화'
#             },
#             '토스': {
#                 'current_valuation': 8000000000000,  # 8조원
#                 'last_funding': '2023-12',
#                 'employees': 5000,
#                 'revenue_estimate': 500000000000,
#                 'recent_news': '금융 슈퍼앱으로 성장, 증권업 진출로 사업 다각화'
#             },
#             '컬리': {
#                 'current_valuation': 2500000000000,  # 2.5조원
#                 'last_funding': '2023-08',
#                 'employees': 2500,
#                 'revenue_estimate': 600000000000,
#                 'recent_news': '새벽배송 시장 선도, 물류 인프라 확장 지속'
#             }
#         }
#         return company_profiles.get(company_name, {})
    
#     def collect_data(self, company_info: CompanyInfo) -> Dict:
#         """비상장기업 데이터 수집"""
#         company_data = self.get_company_info(company_info.name)
#         return company_data

class MarketContextProvider:
    """시장 맥락 정보 제공"""
    
    def __init__(self):
        # 업종별 평균 지표
        self.industry_benchmarks = {
            'KOSPI': {'per': 12.5, 'pbr': 1.1, 'roe': 8.5},
            'KOSDAQ': {'per': 18.0, 'pbr': 1.8, 'roe': 7.2},
            '반도체': {'per': 15.0, 'pbr': 2.0, 'roe': 12.0},
            '인터넷': {'per': 25.0, 'pbr': 3.5, 'roe': 15.0},
            '자동차': {'per': 8.0, 'pbr': 0.8, 'roe': 6.0}
        }
        
        # 비상장 업종별 멀티플
        self.unlisted_multiples = {
            '패션 커머스': {'revenue_multiple': 3.5, 'growth_rate': 25},
            '핀테크': {'revenue_multiple': 8.0, 'growth_rate': 40},
            '부동산테크': {'revenue_multiple': 4.0, 'growth_rate': 30},
            '식품 커머스': {'revenue_multiple': 2.5, 'growth_rate': 20}
        }
    
    def get_market_context(self, company_info: CompanyInfo) -> Dict:
        """시장 맥락 정보 반환"""
        if company_info.is_listed:
            # 상장기업의 경우
            market_avg = self.industry_benchmarks.get(company_info.market, {})
            sector_avg = self.industry_benchmarks.get(company_info.sector, market_avg)
            
            return {
                'industry_per': sector_avg.get('per'),
                'industry_pbr': sector_avg.get('pbr'),
                'industry_roe': sector_avg.get('roe'),
                'market_outlook': '안정적'
            }
        else:
            # 비상장기업의 경우
            multiples = self.unlisted_multiples.get(company_info.sector, {})
            return {
                'sector_multiple': multiples.get('revenue_multiple'),
                'growth_rate': multiples.get('growth_rate'),
                'market_outlook': '성장중'
            }

class CorporateValuationSystem:
    """LLM Agent 기반 기업가치 평가 시스템"""
    
    def __init__(self, api_key: str = None):
        self.llm_agent = LLMAgent(api_key)
        self.nlp_analyzer = NLPAnalyzer(self.llm_agent)
        self.listed_collector = ListedCompanyDataCollector()
        self.unlisted_collector = UnlistedCompanyDataCollector()
        self.market_context = MarketContextProvider()
    
    def analyze(self, query: str) -> str:
        """메인 분석 함수 - LLM Agent 기반"""
        try:
            # 1. LLM을 통한 쿼리 분석
            query_analysis = self.nlp_analyzer.analyze_query(query)
            company_name = query_analysis.get('company_name')
            
            if not company_name:
                return "기업명을 명확히 파악할 수 없습니다. 구체적인 기업명을 포함하여 다시 질문해주세요."
            
            # 2. 기업 분류
            company_info = self.nlp_analyzer.classify_company(company_name)
            
            if company_info.is_listed is None:
                return f"'{company_name}'은 현재 지원하지 않는 기업입니다. 주요 상장기업이나 유명 스타트업명을 입력해주세요."
            
            # 3. 데이터 수집
            if company_info.is_listed:
                financial_data = self.listed_collector.collect_data(company_info)
            else:
                financial_data = self.unlisted_collector.collect_data(company_info)
            
            # 4. 시장 맥락 정보 수집
            market_context = self.market_context.get_market_context(company_info)
            
            # 5. LLM을 통한 종합 분석 리포트 생성
            
            company_info_dict = {
                'name': company_info.name,
                'is_listed': company_info.is_listed,
                'market': company_info.market,
                'sector': company_info.sector
            }
            
            report = self.llm_agent.generate_investment_report(
                query=query,
                company_info=company_info_dict,
                financial_data=financial_data,
                market_context=market_context
            )
            
            return report
            
        except Exception as e:
            return f"분석 중 오류가 발생했습니다: {str(e)}\n\n기술적 문제가 지속되면 잠시 후 다시 시도해주세요."


In [20]:
# 사용 예시
if __name__ == "__main__":
    # 환경변수에 NCP_CLOVASTUDIO_API_KEY 설정 필요
    system = CorporateValuationSystem()
    
    # 테스트 쿼리들
    test_queries = [
        "삼성전자 투자해도 될까?",
        # "무신사 지금 투자 타이밍이야?", 
        # "카카오 주식 어때?",
        # "토스 회사 투자 괜찮을까?"
    ]
    
    for query in test_queries:
        print(f"\n💬 질문: {query}")
        print("="*70)
        try:
            result = system.analyze(query)
            print(result)
        except Exception as e:
            print(f"오류: {e}")
        print("="*70)


💬 질문: 삼성전자 투자해도 될까?
기업명을 명확히 파악할 수 없습니다. 구체적인 기업명을 포함하여 다시 질문해주세요.


In [19]:
# API 연결 여부 확인
import os

# 1. API 키 환경변수 확인
api_key = os.getenv('NCP_CLOVASTUDIO_API_KEY')
print(f"API 키 설정 여부: {'설정됨' if api_key else '설정되지 않음'}")

# 2. langchain_naver 패키지 설치 확인
try:
    from langchain_naver import ChatClovaX
    print("langchain_naver 패키지: 설치됨")
except ImportError:
    print("langchain_naver 패키지: 설치되지 않음")

# 3. 기본 시스템 초기화 테스트
try:
    system = CorporateValuationSystem()
    print("시스템 초기화: 성공")
except Exception as e:
    print(f"시스템 초기화 오류: {e}")

# 4. 간단한 테스트
if 'system' in locals():
    test_result = system.analyze("삼성전자 투자해도 될까?")
    print(f"\n테스트 결과 (첫 200자): {test_result[:200]}...")

API 키 설정 여부: 설정됨
langchain_naver 패키지: 설치됨
시스템 초기화: 성공

테스트 결과 (첫 200자): 기업명을 명확히 파악할 수 없습니다. 구체적인 기업명을 포함하여 다시 질문해주세요....
